# Etap 2. Import danych i czyszczenie

In [1]:
import pandas as pd
customers_raw = pd.read_csv(r'C:\Users\l\Documents\DATA_SCIENCE_KURS\PROJEKT_PREDYCJI_SUBSKRYBCJI_NEWSLETTERA\PROJEKT\b_oprzadek\data\customers.csv')
transactions_raw = pd.read_csv(r'C:\Users\l\Documents\DATA_SCIENCE_KURS\PROJEKT_PREDYCJI_SUBSKRYBCJI_NEWSLETTERA\PROJEKT\b_oprzadek\data\transactions.csv')

#----------KOPIA ZAPASOWA--------------
customers = customers_raw.copy()
transactions = transactions_raw.copy()
#-----------------------------------------

print("----- Raport o stanie danych -----")

print(f"Kształt tabeli customers: {customers.shape}")
print(f"Kształt tabeli transactions: {transactions.shape}")

print("\n===== CUSTOMERS =====")
customers.info()

print("\nOpis:")
print(customers.describe())

print("\nBraki danych:")
print(customers.isna().sum())

print("\nLiczba duplikatów:")
print(customers.duplicated().sum())

print("\n===== TRANSACTIONS =====")
transactions.info()

print("\nOpis:")
print(transactions.describe())

print("\nBraki danych:")
print(transactions.isna().sum())

print("\nLiczba duplikatów:")
print(transactions.duplicated().sum())

----- Raport o stanie danych -----
Kształt tabeli customers: (1010, 9)
Kształt tabeli transactions: (8199, 10)

===== CUSTOMERS =====
<class 'pandas.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   customer_id   1010 non-null   str    
 1   name          1010 non-null   str    
 2   age           954 non-null    float64
 3   gender        1010 non-null   str    
 4   state         956 non-null    str    
 5   signup_date   1010 non-null   str    
 6   email         1010 non-null   str    
 7   phone_number  1010 non-null   int64  
 8   subscribe     1010 non-null   str    
dtypes: float64(1), int64(1), str(7)
memory usage: 71.1 KB

Opis:
              age  phone_number
count  954.000000  1.010000e+03
mean    41.168763  2.130491e+11
std     13.886356  3.435424e+11
min     18.000000  8.001973e+08
25%     29.000000  8.881159e+08
50%     42.000000  8.859298e+09
75%     

In [2]:
print("\n===== CUSTOMERS =====")
print(f"Procent brakujących danych w kolumnie age: {round(customers['age'].isna().sum()/customers.shape[0]*100,2)}%")
print(f"Procent brakujących danych w kolumnie state: {round(customers['state'].isna().sum()/customers.shape[0]*100,2)}%")
print(customers['customer_id'].duplicated().sum())
#SPRAWDZAM CZEMU DUPLIKATY SA W CUSTOMER ID:
duplicates = customers_raw[
    customers_raw["customer_id"].duplicated(keep=False)
].sort_values("customer_id")
print(duplicates)

print("\n===== TRANSACTIONS =====")
print(f"Procent brakujących danych w kolumnie discount_applied: {round(transactions['discount_applied'].isna().sum()/transactions.shape[0]*100,2)}%")
print(f"Procent brakujących danych w kolumnie review_text: {round(transactions['review_text'].isna().sum()/transactions.shape[0]*100,2)}%")


===== CUSTOMERS =====
Procent brakujących danych w kolumnie age: 5.54%
Procent brakujących danych w kolumnie state: 5.35%
10
    customer_id     name   age  gender                      state signup_date  \
141    CUST0137   Faizah  58.0  Female          Sulawesi Tenggara  2025-02-15   
121    CUST0137   Faizah  58.0  Female          Sulawesi Tenggara  2025-02-15   
417    CUST0412     Tira  65.0  Female                       Riau  2024-12-22   
108    CUST0412     Tira  65.0  Female                       Riau  2024-12-22   
20     CUST0514  Adinata  60.0    Male                       Aceh  2023-12-11   
520    CUST0514  Adinata  60.0    Male                       Aceh  2023-12-11   
528    CUST0522   Lurhur  34.0    Male                      Papua  2024-08-14   
104    CUST0522   Lurhur  34.0    Male                      Papua  2024-08-14   
634    CUST0627   Jelita  22.0  Female                       Riau  2023-11-30   
703    CUST0627   Jelita  22.0  Female                       Ria

## Analiza Raportu
Na podstawie raportu można wywnioskować, że:
* w customers:
    * brakuje 56 rekordów klumny 'age':  5.54%,
    * 54 rekordów kolumny 'state':  5.35%,
    * występuje 10 duplikatów (w 100% identycznych)
* w transactions
    * brakuje 426 rekordów kolumny 'discount_applied':  5.2%,
    * 4916 rekordów kolumny 'review_text':  59.96%,
    * nie ma duplikatów

Wszystkie dane mające braki w oklicy 5% należy uzupełnić. Braki w 'review_text' można na razie zostawić bez zmian. W późniejszym etapie można usunąć tę kolumnę całkowicie jeżeli okaże się niepotrzebna.



In [3]:
#Tworzę kopię customers i transactions z nazwą clean do kolejnego etapu
customers_clean = customers.copy()
transactions_clean = transactions.copy()
#Customers age są daną numeryczną co pozwala na zastosowanie mediany.
customers_clean['age'] = customers_clean['age'].fillna(customers_clean['age'].median())
print(customers_clean['age'].isna().sum())
#Customers state są daną kategoryczną więc wypełniam je pozycją "unknown"
customers_clean['state'] = customers_clean['state'].fillna('unknown')
print(customers_clean['state'].isna().sum())
#Usuwam duplikaty
customers_clean = customers.drop_duplicates()
print(customers_clean.duplicated().sum())
#discount_applied jest daną numeryczną, która w momencie braku może przyjąć wartość 0
transactions_clean['discount_applied'] = transactions_clean['discount_applied'].fillna(0)
print(transactions_clean['discount_applied'].isna().sum())

customers_clean.to_csv('customers_clean.csv', index=False)
transactions_clean.to_csv('transactions_clean.csv', index=False)


0
0
0
0


## Podsumowanie

W bazie danych sprzedażowych, tabele customers i transactions miały 3 kolumny z brakami ok. 5% oraz jedną kolumnę ('review_text'), która jest aktualnie w połowie pusta. Braki 5% zostały usupełnione a kolumna 'review_text' nieruszona.